In [ ]:
import os
import time
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

In [ ]:
# ======= Global Config =======
RANDOM_STATE = 42
CLASS_A = 3
CLASS_B = 8
PCA_DIM = 50

RESULT_DIR = "./results_module_b_svm"
FIG_DIR = os.path.join(RESULT_DIR, "figures")
TABLE_DIR = os.path.join(RESULT_DIR, "tables")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

CONFIG = {
    "random_state": RANDOM_STATE,
    "class_a": CLASS_A,
    "class_b": CLASS_B,
    "pca_dim": PCA_DIM,
}

print("Config:")
print(json.dumps(CONFIG, indent=2))

In [ ]:
def load_mnist_binary(class_a=3, class_b=8):
    """
    Load MNIST from OpenML, filter to binary classes (class_a vs class_b),
    normalize pixel values to [0, 1], and preserve official train/test split:
    first 60000 = train, last 10000 = test.
    """
    print("Loading MNIST from OpenML...")
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    
    X = mnist.data.astype(np.float32)
    y = mnist.target.astype(np.int32)
    
    # Normalize to [0, 1]
    X = X / 255.0
    
    # Official split
    X_train_all, X_test_all = X[:60000], X[60000:]
    y_train_all, y_test_all = y[:60000], y[60000:]
    
    # Filter binary classes
    train_mask = (y_train_all == class_a) | (y_train_all == class_b)
    test_mask = (y_test_all == class_a) | (y_test_all == class_b)
    
    X_train = X_train_all[train_mask]
    y_train = y_train_all[train_mask]
    X_test = X_test_all[test_mask]
    y_test = y_test_all[test_mask]
    
    # Map labels to {0, 1}
    y_train_bin = (y_train == class_b).astype(np.int32)
    y_test_bin = (y_test == class_b).astype(np.int32)
    
    print(f"Binary task: {class_a} vs {class_b}")
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    print(f"Train label distribution: {np.bincount(y_train_bin)}  (0->{class_a}, 1->{class_b})")
    print(f"Test  label distribution: {np.bincount(y_test_bin)}  (0->{class_a}, 1->{class_b})")
    
    return X_train, X_test, y_train_bin, y_test_bin

In [ ]:
def apply_pca(X_train, X_test, n_components=50):
    """
    Apply PCA after centering the data.
    Returns transformed train/test data and fitted scaler/pca objects.
    """
    scaler = StandardScaler(with_std=False)
    X_train_centered = scaler.fit_transform(X_train)
    X_test_centered = scaler.transform(X_test)
    
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_centered)
    X_test_pca = pca.transform(X_test_centered)
    
    explained = pca.explained_variance_ratio_.sum()
    print(f"PCA -> n_components={n_components}, explained variance ratio sum={explained:.4f}")
    
    return X_train_pca, X_test_pca, scaler, pca

In [ ]:
def train_svm_model(X_train, y_train, kernel="linear", C=1.0, gamma="scale"):
    """
    Train an SVM model and record training time.
    """
    if kernel == "linear":
        model = SVC(kernel="linear", C=C, random_state=RANDOM_STATE)
    elif kernel == "rbf":
        model = SVC(kernel="rbf", C=C, gamma=gamma, random_state=RANDOM_STATE)
    else:
        raise ValueError(f"Unsupported kernel: {kernel}")
    
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start_time
    
    return model, train_time


def evaluate_model(model, X_test, y_test):
    """
    Evaluate model and record prediction time.
    """
    start_time = time.perf_counter()
    y_pred = model.predict(X_test)
    test_time = time.perf_counter() - start_time
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    # Total number of support vectors
    if hasattr(model, "n_support_"):
        n_support_vectors = int(np.sum(model.n_support_))
    else:
        n_support_vectors = None
    
    report = classification_report(y_test, y_pred, output_dict=True)
    
    metrics = {
        "accuracy": acc,
        "f1": f1,
        "test_time": test_time,
        "n_support_vectors": n_support_vectors,
        "confusion_matrix": cm,
        "classification_report": report,
        "y_pred": y_pred,
    }
    return metrics

In [ ]:
def plot_bar(df, x_col, y_col, title, ylabel, save_path=None, rotation=0):
    plt.figure(figsize=(8, 5))
    plt.bar(df[x_col], df[y_col])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_confusion_matrix(cm, labels, title, save_path=None):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels)
    plt.yticks(tick_marks, labels)
    
    threshold = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > threshold else "black"
            )
    
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = load_mnist_binary(
    class_a=CLASS_A,
    class_b=CLASS_B
)

In [ ]:
X_train_pca, X_test_pca, scaler_pca, pca_model = apply_pca(
    X_train_raw,
    X_test_raw,
    n_components=PCA_DIM
)

In [ ]:
experiments = [
    {
        "name": "Linear SVM + Raw",
        "kernel": "linear",
        "input_type": "Raw",
        "dim": X_train_raw.shape[1],
        "X_train": X_train_raw,
        "X_test": X_test_raw,
        "C": 1.0,
        "gamma": "scale",
    },
    {
        "name": "Linear SVM + PCA",
        "kernel": "linear",
        "input_type": "PCA",
        "dim": X_train_pca.shape[1],
        "X_train": X_train_pca,
        "X_test": X_test_pca,
        "C": 1.0,
        "gamma": "scale",
    },
    {
        "name": "RBF SVM + Raw",
        "kernel": "rbf",
        "input_type": "Raw",
        "dim": X_train_raw.shape[1],
        "X_train": X_train_raw,
        "X_test": X_test_raw,
        "C": 1.0,
        "gamma": "scale",
    },
    {
        "name": "RBF SVM + PCA",
        "kernel": "rbf",
        "input_type": "PCA",
        "dim": X_train_pca.shape[1],
        "X_train": X_train_pca,
        "X_test": X_test_pca,
        "C": 1.0,
        "gamma": "scale",
    },
]

for exp in experiments:
    print(exp["name"], "| dim =", exp["dim"])

In [ ]:
results = []
artifacts = {}

for exp in experiments:
    print("=" * 80)
    print(f"Running: {exp['name']}")
    
    model, train_time = train_svm_model(
        exp["X_train"],
        y_train,
        kernel=exp["kernel"],
        C=exp["C"],
        gamma=exp["gamma"]
    )
    
    metrics = evaluate_model(model, exp["X_test"], y_test)
    
    row = {
        "Model": "Linear SVM" if exp["kernel"] == "linear" else "RBF SVM",
        "Input": exp["input_type"],
        "Dim": exp["dim"],
        "Accuracy": metrics["accuracy"],
        "F1": metrics["f1"],
        "Train Time (s)": train_time,
        "Test Time (s)": metrics["test_time"],
        "#Support Vectors": metrics["n_support_vectors"],
    }
    results.append(row)
    
    artifacts[exp["name"]] = {
        "model": model,
        "metrics": metrics,
        "config": exp,
        "train_time": train_time,
    }
    
    print(pd.Series(row))

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Model", "Input"]).reset_index(drop=True)

results_df

In [ ]:
csv_path = os.path.join(TABLE_DIR, "module_b_svm_results.csv")
results_df.to_csv(csv_path, index=False)

print(f"Saved results table to: {csv_path}")

In [ ]:
plot_df = results_df.copy()
plot_df["Label"] = plot_df["Model"] + " + " + plot_df["Input"]

plot_bar(
    plot_df,
    x_col="Label",
    y_col="Accuracy",
    title=f"Accuracy Comparison ({CLASS_A} vs {CLASS_B})",
    ylabel="Accuracy",
    save_path=os.path.join(FIG_DIR, "accuracy_bar.png"),
    rotation=15
)

In [ ]:
plot_bar(
    plot_df,
    x_col="Label",
    y_col="Train Time (s)",
    title=f"Training Time Comparison ({CLASS_A} vs {CLASS_B})",
    ylabel="Train Time (s)",
    save_path=os.path.join(FIG_DIR, "train_time_bar.png"),
    rotation=15
)

In [ ]:
plot_bar(
    plot_df,
    x_col="Label",
    y_col="#Support Vectors",
    title=f"Number of Support Vectors ({CLASS_A} vs {CLASS_B})",
    ylabel="#Support Vectors",
    save_path=os.path.join(FIG_DIR, "support_vectors_bar.png"),
    rotation=15
)

In [ ]:
cm_targets = [
    "Linear SVM + Raw",
    "RBF SVM + Raw",
]

for name in cm_targets:
    cm = artifacts[name]["metrics"]["confusion_matrix"]
    plot_confusion_matrix(
        cm,
        labels=[str(CLASS_A), str(CLASS_B)],
        title=f"Confusion Matrix - {name}",
        save_path=os.path.join(FIG_DIR, f"cm_{name.lower().replace(' ', '_').replace('+', 'plus')}.png")
    )

In [ ]:
for name in artifacts:
    print("=" * 80)
    print(name)
    print(pd.DataFrame(artifacts[name]["metrics"]["classification_report"]).T)

In [ ]:
best_acc_idx = results_df["Accuracy"].idxmax()
best_row = results_df.loc[best_acc_idx]

summary_text = f"""
模块B实验采用MNIST数据集上的二分类任务（{CLASS_A} vs {CLASS_B}），
比较了线性核SVM与RBF核SVM在原始784维像素输入和PCA降维输入（{PCA_DIM}维）下的分类表现。

从实验结果看，最佳模型为 {best_row['Model']} + {best_row['Input']}，
其准确率为 {best_row['Accuracy']:.4f}，F1值为 {best_row['F1']:.4f}。
同时，PCA预处理通常能显著降低训练时间，
但是否保持精度则需要结合具体模型结果分析。

线性SVM能够提供一个清晰的线性分类基线，
而RBF核SVM能够进一步利用非线性结构提升分类性能。
支持向量数量则反映了模型决策边界的复杂程度，
可作为分析模型复杂性的辅助指标。
"""

print(summary_text)

In [ ]:
tasks = [
    (3, 8),
    (4, 9),
    (1, 7),
]

all_results = []

for class_a, class_b in tasks:
    print("=" * 100)
    print(f"Task: {class_a} vs {class_b}")
    
    # 重新加载数据
    X_train_raw, X_test_raw, y_train, y_test = load_mnist_binary(
        class_a=class_a,
        class_b=class_b
    )
    
    # PCA
    X_train_pca, X_test_pca, _, _ = apply_pca(
        X_train_raw,
        X_test_raw,
        n_components=PCA_DIM
    )
    
    # 重建实验列表（用当前数据）
    experiments = [
        ("Linear SVM", "Raw", "linear", X_train_raw, X_test_raw),
        ("Linear SVM", "PCA", "linear", X_train_pca, X_test_pca),
        ("RBF SVM", "Raw", "rbf", X_train_raw, X_test_raw),
        ("RBF SVM", "PCA", "rbf", X_train_pca, X_test_pca),
    ]
    
    for model_name, input_type, kernel, X_tr, X_te in experiments:
        model, train_time = train_svm_model(X_tr, y_train, kernel=kernel)
        metrics = evaluate_model(model, X_te, y_test)
        
        row = {
            "Task": f"{class_a} vs {class_b}",
            "Model": model_name,
            "Input": input_type,
            "Accuracy": metrics["accuracy"],
            "F1": metrics["f1"],
            "Train Time": train_time,
            "#SV": metrics["n_support_vectors"],
        }
        
        all_results.append(row)

all_results_df = pd.DataFrame(all_results)
all_results_df

In [ ]:
# =======================
# Part B: Multiclass SVM (RBF + PCA50) : OvO vs OvR
# with Progress & Timing
# =======================

import itertools
import time
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.base import clone

# ---------- config ----------
RANDOM_STATE = 42
PCA_DIM = 50
C_VALUE = 5.0
GAMMA_VALUE = "scale"

# ---------- utils ----------
def format_seconds(seconds):
    seconds = float(seconds)
    if seconds < 60:
        return f"{seconds:.1f}s"
    m = int(seconds // 60)
    s = seconds % 60
    return f"{m}m {s:.1f}s"

def load_mnist_multiclass():
    print("Loading MNIST (10-class) from OpenML...")
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    X = mnist.data.astype(np.float32) / 255.0
    y = mnist.target.astype(np.int32)

    # official split
    X_train, X_test = X[:60000], X[60000:]
    y_train, y_test = y[:60000], y[60000:]

    print(f"Train: {X_train.shape}, Test: {X_test.shape}")
    return X_train, X_test, y_train, y_test

def apply_pca(X_train, X_test, n_components=50):
    """
    Center data first, then PCA.
    """
    print(f"Applying PCA(n_components={n_components}) ...")
    scaler = StandardScaler(with_std=False)
    X_train_centered = scaler.fit_transform(X_train)
    X_test_centered = scaler.transform(X_test)

    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_centered)
    X_test_pca = pca.transform(X_test_centered)

    explained = pca.explained_variance_ratio_.sum()
    print(f"PCA done. Explained variance ratio sum = {explained:.4f}")
    print(f"Train PCA shape: {X_train_pca.shape}, Test PCA shape: {X_test_pca.shape}")

    return X_train_pca, X_test_pca, scaler, pca

def train_multiclass_svm_with_progress(
    X_train, y_train, strategy="ovo", kernel="rbf", C=5.0, gamma="scale", experiment_name=None
):
    classes = np.sort(np.unique(y_train))
    n_classes = len(classes)

    base_svc = SVC(
        kernel=kernel,
        C=C,
        gamma=gamma,
        random_state=RANDOM_STATE
    )

    overall_start = time.perf_counter()

    if strategy == "ovo":
        class_pairs = list(itertools.combinations(classes, 2))
        total_models = len(class_pairs)  # 10 classes -> 45
        estimators = []

        print("=" * 100)
        print(f"[{experiment_name}] strategy=OvO, kernel={kernel}, total binary classifiers = {total_models}")

        for idx, (ci, cj) in enumerate(class_pairs, start=1):
            pair_start = time.perf_counter()

            mask = (y_train == ci) | (y_train == cj)
            X_pair = X_train[mask]
            y_pair = y_train[mask]

            est = clone(base_svc)
            est.fit(X_pair, y_pair)

            pair_time = time.perf_counter() - pair_start
            elapsed = time.perf_counter() - overall_start
            avg_time = elapsed / idx
            eta = avg_time * (total_models - idx)

            estimators.append({
                "classes": (ci, cj),
                "estimator": est
            })

            print(
                f"[{experiment_name}] OvO progress: {idx:02d}/{total_models} "
                f"(class {ci} vs {cj}) | this={format_seconds(pair_time)} | "
                f"elapsed={format_seconds(elapsed)} | eta={format_seconds(eta)}"
            )

        train_time = time.perf_counter() - overall_start
        model = {"strategy": "ovo", "classes": classes, "estimators": estimators}
        return model, train_time

    elif strategy == "ovr":
        total_models = n_classes
        estimators = []

        print("=" * 100)
        print(f"[{experiment_name}] strategy=OvR, kernel={kernel}, total binary classifiers = {total_models}")

        for idx, c in enumerate(classes, start=1):
            clf_start = time.perf_counter()

            y_bin = (y_train == c).astype(np.int32)
            est = clone(base_svc)
            est.fit(X_train, y_bin)

            clf_time = time.perf_counter() - clf_start
            elapsed = time.perf_counter() - overall_start
            avg_time = elapsed / idx
            eta = avg_time * (total_models - idx)

            estimators.append({
                "class": c,
                "estimator": est
            })

            print(
                f"[{experiment_name}] OvR progress: {idx:02d}/{total_models} "
                f"(class {c} vs rest) | this={format_seconds(clf_time)} | "
                f"elapsed={format_seconds(elapsed)} | eta={format_seconds(eta)}"
            )

        train_time = time.perf_counter() - overall_start
        model = {"strategy": "ovr", "classes": classes, "estimators": estimators}
        return model, train_time

    else:
        raise ValueError("Unsupported strategy")

def predict_multiclass_model(model, X):
    strategy = model["strategy"]
    classes = model["classes"]

    if strategy == "ovo":
        n_samples = X.shape[0]
        n_classes = len(classes)
        class_to_index = {c: i for i, c in enumerate(classes)}
        votes = np.zeros((n_samples, n_classes), dtype=np.int32)

        for item in model["estimators"]:
            ci, cj = item["classes"]
            est = item["estimator"]
            pred = est.predict(X)
            votes[:, class_to_index[ci]] += (pred == ci)
            votes[:, class_to_index[cj]] += (pred == cj)

        return classes[np.argmax(votes, axis=1)]

    elif strategy == "ovr":
        scores = []
        for item in model["estimators"]:
            est = item["estimator"]
            if hasattr(est, "decision_function"):
                score = est.decision_function(X)
            else:
                score = est.predict(X)
            scores.append(score)
        scores = np.column_stack(scores)
        return classes[np.argmax(scores, axis=1)]

def evaluate_multiclass_model(model, X_test, y_test):
    start = time.perf_counter()
    y_pred = predict_multiclass_model(model, X_test)
    test_time = time.perf_counter() - start

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "test_time": test_time,
        "confusion_matrix": cm,
        "classification_report": report,
    }

# ---------- load data ----------
X_train_raw, X_test_raw, y_train_mc, y_test_mc = load_mnist_multiclass()

# ---------- PCA ----------
X_train_mc, X_test_mc, scaler_pca, pca_model = apply_pca(
    X_train_raw, X_test_raw, n_components=PCA_DIM
)

# ---------- experiments ----------
experiments = [
    {"name": "RBF SVM + PCA50 + OvO", "strategy": "ovo", "kernel": "rbf", "C": C_VALUE, "gamma": GAMMA_VALUE},
    {"name": "RBF SVM + PCA50 + OvR", "strategy": "ovr", "kernel": "rbf", "C": C_VALUE, "gamma": GAMMA_VALUE},
]

# ---------- run ----------
results = []
artifacts = {}
all_start = time.perf_counter()

for i, exp in enumerate(experiments, start=1):
    exp_start = time.perf_counter()

    print("\n" + "#" * 120)
    print(f"MAIN EXP {i}/{len(experiments)}: {exp['name']}")
    print("#" * 120)

    model, train_time = train_multiclass_svm_with_progress(
        X_train_mc,
        y_train_mc,
        strategy=exp["strategy"],
        kernel=exp["kernel"],
        C=exp["C"],
        gamma=exp["gamma"],
        experiment_name=exp["name"]
    )

    print(f"[{exp['name']}] evaluating...")

    metrics = evaluate_multiclass_model(model, X_test_mc, y_test_mc)

    exp_time = time.perf_counter() - exp_start
    elapsed = time.perf_counter() - all_start
    avg = elapsed / i
    eta = avg * (len(experiments) - i)

    results.append({
        "Model": exp["name"],
        "Accuracy": metrics["accuracy"],
        "Macro F1": metrics["macro_f1"],
        "Train Time (s)": train_time,
        "Test Time (s)": metrics["test_time"],
        "Total Time (s)": exp_time
    })

    artifacts[exp["name"]] = {
        "model": model,
        "metrics": metrics
    }

    print(f"[{exp['name']}] DONE | acc={metrics['accuracy']:.4f} | "
          f"macro_f1={metrics['macro_f1']:.4f} | "
          f"elapsed={format_seconds(elapsed)} | eta={format_seconds(eta)}")

# ---------- final results ----------
results_df = pd.DataFrame(results)
print("\nFinal Results:")
display(results_df)

# ---------- optional quick look at confusion matrices ----------
for name in artifacts:
    print("=" * 100)
    print(name)
    print("Confusion matrix:")
    print(artifacts[name]["metrics"]["confusion_matrix"])

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

# ===== report-friendly figure dir =====
FIG_DIR = "./results_module_b_svm/figures_multiclass"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["font.size"] = 11

In [ ]:
def save_show(save_path=None):
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()


def plot_metric_bars(results_df, save_dir=FIG_DIR):
    df = results_df.copy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df["Model"], df["Accuracy"])
    ax.set_title("Accuracy comparison of OvO and OvR")
    ax.set_ylabel("Accuracy")
    ax.tick_params(axis="x", rotation=12)
    save_show(os.path.join(save_dir, "b1_accuracy_comparison.png"))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df["Model"], df["Macro F1"])
    ax.set_title("Macro-F1 comparison of OvO and OvR")
    ax.set_ylabel("Macro-F1")
    ax.tick_params(axis="x", rotation=12)
    save_show(os.path.join(save_dir, "b2_macro_f1_comparison.png"))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df["Model"], df["Train Time (s)"])
    ax.set_title("Training time comparison of OvO and OvR")
    ax.set_ylabel("Train Time (s)")
    ax.tick_params(axis="x", rotation=12)
    save_show(os.path.join(save_dir, "b3_train_time_comparison.png"))


def plot_confusion_matrix_pretty(cm, class_labels, title, save_path=None):
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)

    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))
    ax.set_xticklabels(class_labels)
    ax.set_yticklabels(class_labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, f"{cm[i, j]}",
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=9
            )

    save_show(save_path)

In [ ]:
display(results_df)
plot_metric_bars(results_df, save_dir=FIG_DIR)

In [ ]:
class_labels = list(range(10))

for model_name, item in artifacts.items():
    cm = item["metrics"]["confusion_matrix"]
    plot_confusion_matrix_pretty(
        cm,
        class_labels=class_labels,
        title=f"Confusion Matrix: {model_name}",
        save_path=os.path.join(
            FIG_DIR,
            f"cm_{model_name.lower().replace(' ', '_').replace('+', 'plus')}.png"
        )
    )

In [ ]:
def get_top_confused_pairs(cm, top_k=10):
    pairs = []
    n = cm.shape[0]
    for i in range(n):
        for j in range(n):
            if i != j:
                pairs.append((i, j, cm[i, j]))
    pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
    return pairs[:top_k]


def plot_top_confused_pairs(cm, model_name, top_k=10, save_dir=FIG_DIR):
    pairs = get_top_confused_pairs(cm, top_k=top_k)
    labels = [f"{i}->{j}" for i, j, _ in pairs]
    values = [v for _, _, v in pairs]

    fig, ax = plt.subplots(figsize=(8.5, 5))
    ax.bar(labels, values)
    ax.set_title(f"Top-{top_k} confusion pairs: {model_name}")
    ax.set_ylabel("Number of misclassified samples")
    ax.tick_params(axis="x", rotation=35)
    save_show(os.path.join(
        save_dir,
        f"top_confusions_{model_name.lower().replace(' ', '_').replace('+', 'plus')}.png"
    ))

    return pd.DataFrame(pairs, columns=["true_class", "pred_class", "count"])


for model_name, item in artifacts.items():
    cm = item["metrics"]["confusion_matrix"]
    top_conf_df = plot_top_confused_pairs(cm, model_name, top_k=10, save_dir=FIG_DIR)
    print(f"\nTop confused pairs for {model_name}:")
    display(top_conf_df)

In [ ]:
def summarize_ovo_support_vectors(model_dict):
    rows = []
    for item in model_dict["estimators"]:
        ci, cj = item["classes"]
        est = item["estimator"]
        n_sv_total = int(np.sum(est.n_support_))
        rows.append({
            "pair": f"{ci} vs {cj}",
            "class_i": ci,
            "class_j": cj,
            "n_sv_total": n_sv_total,
            "n_sv_class_i": int(est.n_support_[0]),
            "n_sv_class_j": int(est.n_support_[1]),
        })
    return pd.DataFrame(rows).sort_values("n_sv_total", ascending=False).reset_index(drop=True)


ovo_name = [k for k in artifacts.keys() if "OvO" in k][0]
ovo_model = artifacts[ovo_name]["model"]
ovo_sv_df = summarize_ovo_support_vectors(ovo_model)

display(ovo_sv_df.head(15))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(ovo_sv_df["pair"], ovo_sv_df["n_sv_total"])
ax.set_title(f"Support-vector counts across 45 OvO binary classifiers")
ax.set_ylabel("Number of support vectors")
ax.tick_params(axis="x", rotation=90)
save_show(os.path.join(FIG_DIR, "support_vectors_ovo_all_pairs.png"))

In [ ]:
def summarize_ovr_support_vectors(model_dict):
    rows = []
    for item in model_dict["estimators"]:
        c = item["class"]
        est = item["estimator"]
        n_sv_total = int(np.sum(est.n_support_))
        rows.append({
            "class": int(c),
            "n_sv_total": n_sv_total,
            "n_sv_negative": int(est.n_support_[0]),
            "n_sv_positive": int(est.n_support_[1]),
        })
    return pd.DataFrame(rows).sort_values("class").reset_index(drop=True)


ovr_name = [k for k in artifacts.keys() if "OvR" in k][0]
ovr_model = artifacts[ovr_name]["model"]
ovr_sv_df = summarize_ovr_support_vectors(ovr_model)

display(ovr_sv_df)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(ovr_sv_df["class"].astype(str), ovr_sv_df["n_sv_total"])
ax.set_title("Support-vector counts across 10 OvR classifiers")
ax.set_xlabel("Class (positive class vs rest)")
ax.set_ylabel("Number of support vectors")
save_show(os.path.join(FIG_DIR, "support_vectors_ovr_per_class.png"))

In [ ]:
cum_var = np.cumsum(pca_model.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(np.arange(1, len(cum_var) + 1), cum_var, marker="o", markersize=3)
ax.axhline(cum_var[-1], linestyle="--")
ax.set_title("Cumulative explained variance of PCA components")
ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative explained variance ratio")
save_show(os.path.join(FIG_DIR, "pca50_cumulative_explained_variance.png"))

print(f"Cumulative explained variance at PCA-50: {cum_var[-1]:.4f}")

In [ ]:
summary_rows = []

for model_name, item in artifacts.items():
    cm = item["metrics"]["confusion_matrix"]
    acc = item["metrics"]["accuracy"]
    macro_f1 = item["metrics"]["macro_f1"]

    top_pairs = get_top_confused_pairs(cm, top_k=3)
    top_pairs_text = "; ".join([f"{i}->{j}:{v}" for i, j, v in top_pairs])

    summary_rows.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "top_3_confusions": top_pairs_text,
    })

summary_vis_df = pd.DataFrame(summary_rows)
display(summary_vis_df)

summary_vis_df.to_csv(
    os.path.join(FIG_DIR, "visual_summary_table.csv"),
    index=False,
    encoding="utf-8-sig"
)